# ConfigLoader Example

This notebook reads shared example config files from `docs/examples/system/config` and system-specific parameter files from `docs/examples/system/params/<system_id>`, then loads them with `ConfigLoader`.


In [ ]:
from pathlib import Path

import yaml

from qubex.system import ConfigLoader

## Resolve example paths

Choose one of `64Q-HF-Q1`, `144Q-LF-Q1`, or `144Q-LF-Q3`, and the notebook resolves the shared config directory together with the matching parameter directory.

This supports running the notebook from the repository root, `docs/examples/system`, or one of its child directories.


In [ ]:
system_id = "144Q-LF-Q3"
supported_system_ids = {"64Q-HF-Q1", "144Q-LF-Q1", "144Q-LF-Q3"}
if system_id not in supported_system_ids:
    raise ValueError(f"Unsupported system_id: {system_id}")

candidate_root_dirs = [
    Path.cwd(),
    Path.cwd() / "docs/examples/system",
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

example_root = next(
    (
        path
        for path in candidate_root_dirs
        if (path / "config").is_dir() and (path / "params").is_dir()
    ),
    None,
)
if example_root is None:
    raise FileNotFoundError(
        "Could not find `docs/examples/system/{config,params}` from the current working directory."
    )

config_dir = example_root / "config"
params_dir = example_root / "params" / system_id

print(f"example_root: {example_root.resolve()}")
print(f"system_id   : {system_id}")
print(f"config_dir  : {config_dir.resolve()}")
print(f"params_dir  : {params_dir.resolve()}")


## Read config files directly


In [ ]:
config_payloads = {}
for path in sorted(config_dir.glob("*.yaml")):
    with path.open() as f:
        config_payloads[path.name] = yaml.safe_load(f)

system_entry = config_payloads["system.yaml"][system_id]

print("loaded files:", ", ".join(config_payloads))
print("system backend:", system_entry["backend"])
print("chip id:", system_entry["chip_id"])


## Load the same files with `ConfigLoader`


In [ ]:
loader = ConfigLoader(
    system_id=system_id,
    config_dir=config_dir,
    params_dir=params_dir,
    autoload=False,
)
loader.load()
system = loader.get_experiment_system()

print("system_id    :", loader.system_id)
print("chip_id      :", loader.chip_id)
print("backend_kind :", loader.backend_kind)
print("wiring_file  :", loader.wiring_file)
print("qubits       :", len(system.qubits))
print("resonators   :", len(system.resonators))
print("boxes        :", len(system.boxes))


## Access loaded parameters


In [ ]:
first_qubit = system.qubits[0].label
print("first qubit label      :", first_qubit)
print("control amplitude      :", system.control_params.get_control_amplitude(first_qubit))
print("readout amplitude      :", system.control_params.get_readout_amplitude(first_qubit))
print("MUX0 capture delay     :", system.control_params.get_capture_delay(0))
